<a href="https://colab.research.google.com/github/khodiboev/computer_vision/blob/main/data_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Scraping & Cleaning — Kebab Images for the Menu Detector

Food-101 has no `kebab` class, so kebab images were scraped from Bing and added to
`food101_dataset/kebab` (used by `menu_detector_model.ipynb`).

**Pipeline**
1. Download images from Bing with several kebab-related queries
2. Remove broken / tiny images and near-duplicates (perceptual hash), also against images already in the dataset
3. **CLIP zero-shot filtering**: score every image for "kebab" vs. noise (logos, menus, people, buildings, drawings)
4. Manual review of the remaining images (sorted from least to most confident)
5. Copy the approved images to Google Drive (copy only — nothing in Drive is deleted)

**Result of the original run (Sep 2026)**

| Stage | Images |
|---|---|
| Raw download (7 queries) | 478 |
| After broken/tiny/duplicate removal | 466 |
| CLIP kebab score ≥ 0.8 | 36 |
| After manual review | **24** |

Lesson learned: ~90% of the scraped images were unrelated noise (queries that returned the full 100 images
were almost entirely off-topic), so automatic CLIP filtering plus a quick manual check was essential.
The kebab class grew from 89 to 113 images.

Runtime: CPU is enough (GPU makes the CLIP step faster).

In [ ]:
# ----------------------------
# Setup & configuration
# ----------------------------
!pip install -q bing-image-downloader imagehash

import os
import shutil

import pandas as pd
import matplotlib.pyplot as plt
import torch
import imagehash
from PIL import Image
from bing_image_downloader import downloader
from transformers import CLIPProcessor, CLIPModel
from google.colab import drive

drive.mount('/content/drive')

RAW_DIR = '/content/kebab_raw'
CLEAN_DIR = '/content/kebab_clean'
TARGET_DIR = '/content/drive/MyDrive/Colab Notebooks/food101_dataset/kebab'

QUERIES = [
    'shish kebab skewers',
    'doner kebab plate',
    'adana kebab',
    'shashlik grilled meat skewers',
    'chicken kebab plate',
    'lamb kebab grill',
    'iskender kebab',
]
LIMIT_PER_QUERY = 100
MIN_SIDE = 150          # smaller images are usually icons / thumbnails
HASH_DISTANCE = 5       # perceptual-hash distance treated as a duplicate
CLIP_THRESHOLD = 0.8    # minimum "kebab" probability to keep an image

In [ ]:
# ----------------------------
# 1. Download from Bing
# ----------------------------
for q in QUERIES:
    print(f'>>> {q}')
    downloader.download(q, limit=LIMIT_PER_QUERY, output_dir=RAW_DIR,
                        adult_filter_off=False, force_replace=False, timeout=30, verbose=False)

print('\n=== Downloaded ===')
for q in QUERIES:
    folder = os.path.join(RAW_DIR, q)
    print(f'{q:35s} {len(os.listdir(folder)) if os.path.exists(folder) else 0:4d}')

In [ ]:
# ----------------------------
# 2. Remove broken, tiny and duplicate images
# ----------------------------
os.makedirs(CLEAN_DIR, exist_ok=True)

# hashes of images already in the dataset, so we don't add copies of them
seen_hashes = []
for fname in os.listdir(TARGET_DIR):
    try:
        with Image.open(os.path.join(TARGET_DIR, fname)) as img:
            seen_hashes.append(imagehash.phash(img.convert('RGB')))
    except Exception:
        pass
print('Existing images hashed:', len(seen_hashes))

stats = {'total': 0, 'broken': 0, 'too_small': 0, 'duplicate': 0, 'kept': 0}
records = []

for q in QUERIES:
    folder = os.path.join(RAW_DIR, q)
    if not os.path.isdir(folder):
        continue
    for fname in sorted(os.listdir(folder)):
        stats['total'] += 1
        path = os.path.join(folder, fname)
        try:
            with Image.open(path) as img:
                img.verify()
            img = Image.open(path).convert('RGB')
        except Exception:
            stats['broken'] += 1
            continue
        if min(img.size) < MIN_SIDE:
            stats['too_small'] += 1
            continue
        h = imagehash.phash(img)
        if any(h - s <= HASH_DISTANCE for s in seen_hashes):
            stats['duplicate'] += 1
            continue
        seen_hashes.append(h)

        stats['kept'] += 1
        out_name = f"kebab_bing_{stats['kept']:04d}.jpg"
        img.save(os.path.join(CLEAN_DIR, out_name), 'JPEG', quality=92)
        records.append({'file': out_name, 'query': q})

clean_df = pd.DataFrame(records)
print(pd.Series(stats))

In [ ]:
# ----------------------------
# 3. CLIP zero-shot filtering
# ----------------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device).eval()
clip_proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

TEXTS = [
    'a photo of kebab, grilled meat dish',   # index 0 = the class we want
    'a logo, text, or restaurant menu',
    'a photo of a person',
    'a photo of a restaurant building or street',
    'a cartoon or drawing',
]

scores = []
with torch.no_grad():
    for i in range(0, len(clean_df), 32):
        batch = clean_df.file.iloc[i:i + 32]
        imgs = [Image.open(os.path.join(CLEAN_DIR, f)).convert('RGB') for f in batch]
        inputs = clip_proc(text=TEXTS, images=imgs, return_tensors='pt', padding=True).to(device)
        probs = clip_model(**inputs).logits_per_image.softmax(dim=-1)
        scores.extend(probs[:, 0].cpu().tolist())
clean_df['kebab_score'] = scores

per_query = clean_df.groupby('query').agg(
    images=('file', 'size'),
    kept=('kebab_score', lambda s: (s >= CLIP_THRESHOLD).sum()),
)
per_query['share'] = (per_query.kept / per_query.images).map('{:.0%}'.format)
print(per_query)
print(f'\nScore >= {CLIP_THRESHOLD}: {(clean_df.kebab_score >= CLIP_THRESHOLD).sum()} of {len(clean_df)}')

In [ ]:
# ----------------------------
# 4. Manual review of CLIP-approved images (least confident first)
# ----------------------------
candidates = (clean_df[clean_df.kebab_score >= CLIP_THRESHOLD]
              .sort_values('kebab_score').reset_index(drop=True))
candidates.index = candidates.index + 1

cols = 8
rows = (len(candidates) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.5))
for ax in axes.flat:
    ax.axis('off')
for ax, (idx, row) in zip(axes.flat, candidates.iterrows()):
    ax.imshow(Image.open(os.path.join(CLEAN_DIR, row.file)))
    ax.set_title(f'N{idx}  {row.kebab_score:.2f}', fontsize=9)
plt.suptitle('CLIP-approved images — note the N numbers that are NOT kebab', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------
# 5. Apply manual rejections and copy to Drive
# ----------------------------
# Fill in the N numbers of images that are not kebab.
# In the original run: 1-11 and 13 (grilled chicken, steak, rack of lamb, stew, a book cover).
MANUAL_REJECT = []

approved = candidates.drop(index=[n for n in MANUAL_REJECT if n in candidates.index])

copied = 0
for f in approved.file:
    dst = os.path.join(TARGET_DIR, f)
    if not os.path.exists(dst):
        shutil.copy2(os.path.join(CLEAN_DIR, f), dst)
        copied += 1

print(f'Approved: {len(approved)} | copied to Drive: {copied}')
print(f'Kebab folder now contains: {len(os.listdir(TARGET_DIR))} files')